EDA on PR Data 

In [0]:
VOLUME_PATH= "/Volumes/pr_population_volume/default/raw"

spark.sql(f"LIST '{VOLUME_PATH}/data/pr_data'").display()


path,name,size,modification_time
/Volumes/pr_population_volume/default/raw/data/pr_data/pr.class,pr.class,102,1791328422000
/Volumes/pr_population_volume/default/raw/data/pr_data/pr.contacts,pr.contacts,562,1791328424000
/Volumes/pr_population_volume/default/raw/data/pr_data/pr.data.0.Current,pr.data.0.Current,1615931,1791328426000
/Volumes/pr_population_volume/default/raw/data/pr_data/pr.data.1.AllData,pr.data.1.AllData,3239525,1791328427000
/Volumes/pr_population_volume/default/raw/data/pr_data/pr.duration,pr.duration,176,1791328429000
/Volumes/pr_population_volume/default/raw/data/pr_data/pr.footnote,pr.footnote,40,1791328431000
/Volumes/pr_population_volume/default/raw/data/pr_data/pr.measure,pr.measure,745,1791328432000
/Volumes/pr_population_volume/default/raw/data/pr_data/pr.period,pr.period,146,1791328434000
/Volumes/pr_population_volume/default/raw/data/pr_data/pr.seasonal,pr.seasonal,79,1791328435000
/Volumes/pr_population_volume/default/raw/data/pr_data/pr.sector,pr.sector,263,1791328437000


In [0]:
#/Volumes/pr_population_volume/default/raw/data/pr_data/pr.data.0.Current

BASE_DIR="/Volumes/pr_population_volume/default/raw/data"
schema = spark.read.format("csv").options(header=True,inferSchema=True,seperator='\t').load(f"{BASE_DIR}/pr_data/pr.data.0.Current").schema
schema

StructType([StructField('series_id        	year	period	       value	footnote_codes', StringType(), True)])

In [0]:
spark.sql(f"LIST '{VOLUME_PATH}/data/us_population'").display()

path,name,size,modification_time
/Volumes/pr_population_volume/default/raw/data/us_population/population.csv,population.csv,462,1791328480000


In [0]:
%sql 
SHOW SCHEMAS IN pr_population_volume;

databaseName
bronze
default
gold
information_schema
silver


In [0]:
BASE_DIR="/Volumes/pr_population_volume/default/raw/data"
schema = spark.read.format("csv").options(header=True,inferSchema=True).load(f"{BASE_DIR}/us_population/population.csv").schema
schema



StructType([StructField('Nation ID', StringType(), True), StructField('Nation', StringType(), True), StructField('Year', IntegerType(), True), StructField('Population', DoubleType(), True)])

In [0]:
BASE_DIR = "/Volumes/pr_population_volume/default/raw/data"
schema = (
    spark.read.format("json")
    .options(inferSchema=True)
    .load(f"{BASE_DIR}/us_population/population.json")
    .schema
)

schema

StructType([StructField('annotations', StructType([StructField('dataset_link', StringType(), True), StructField('dataset_name', StringType(), True), StructField('source_description', StringType(), True), StructField('source_name', StringType(), True), StructField('subtopic', StringType(), True), StructField('table_id', StringType(), True), StructField('topic', StringType(), True)]), True), StructField('columns', ArrayType(StringType(), True), True), StructField('data', ArrayType(StructType([StructField('Nation', StringType(), True), StructField('Nation ID', StringType(), True), StructField('Population', DoubleType(), True), StructField('Year', LongType(), True)]), True), True), StructField('page', StructType([StructField('limit', LongType(), True), StructField('offset', LongType(), True), StructField('total', LongType(), True)]), True)])

In [0]:
%sql
select * from pr_population_volume.bronze.us_population;

annotations,columns,data,page
null,null,null,null
null,null,null,null
null,null,null,null
null,null,null,null
null,null,null,null
null,null,null,null
null,null,null,null
null,null,null,null
null,null,null,null
null,null,null,null


In [0]:
{
    "annotations": {
        "source_name": "Census Bureau",
        "subtopic": "Demographics",
        "dataset_name": "ACS 1-year Estimate",
        "topic": "Diversity",
        "table_id": "B01003",
        "dataset_link": "http://www.census.gov/programs-surveys/acs/",
        "source_description": "The American Community Survey (ACS) is conducted by the US Census and sent to a portion of the population every year.",
    },
    "page": {"limit": 0, "offset": 0, "total": 11},
    "columns": ["Nation ID", "Nation", "Year", "Population"],
    "data": [
        {
            "Nation ID": "01000US",
            "Nation": "United States",
            "Year": 2013,
            "Population": 316128839.0,
        },
        {
            "Nation ID": "01000US",
            "Nation": "United States",
            "Year": 2014,
            "Population": 318857056.0,
        },
        {
            "Nation ID": "01000US",
            "Nation": "United States",
            "Year": 2015,
            "Population": 321418821.0,
        },
        {
            "Nation ID": "01000US",
            "Nation": "United States",
            "Year": 2016,
            "Population": 323127515.0,
        },
        {
            "Nation ID": "01000US",
            "Nation": "United States",
            "Year": 2017,
            "Population": 325719178.0,
        },
        {
            "Nation ID": "01000US",
            "Nation": "United States",
            "Year": 2018,
            "Population": 327167439.0,
        },
        {
            "Nation ID": "01000US",
            "Nation": "United States",
            "Year": 2019,
            "Population": 328239523.0,
        },
        {
            "Nation ID": "01000US",
            "Nation": "United States",
            "Year": 2021,
            "Population": 331893745.0,
        },
        {
            "Nation ID": "01000US",
            "Nation": "United States",
            "Year": 2022,
            "Population": 333287562.0,
        },
        {
            "Nation ID": "01000US",
            "Nation": "United States",
            "Year": 2023,
            "Population": 334914896.0,
        },
        {
            "Nation ID": "01000US",
            "Nation": "United States",
            "Year": 2024,
            "Population": 340110990.0,
        },
    ],
}

In [0]:
from pyspark.sql.functions import explode, col

# 1. Read the JSON file
df = spark.read.json("/Volumes/pr_population_volume/default/raw/data/us_population/population.json")

# 2. Explode the "data" array
exploded_df = df.select(explode(col("data")).alias("data"))
#exploded_df.show()
# 3. Flatten the struct inside "data"
flattened_df = exploded_df.select("data.*")

flattened_df.show()



+-------------+---------+------------+----+
|       Nation|Nation ID|  Population|Year|
+-------------+---------+------------+----+
|United States|  01000US|3.16128839E8|2013|
|United States|  01000US|3.18857056E8|2014|
|United States|  01000US|3.21418821E8|2015|
|United States|  01000US|3.23127515E8|2016|
|United States|  01000US|3.25719178E8|2017|
|United States|  01000US|3.27167439E8|2018|
|United States|  01000US|3.28239523E8|2019|
|United States|  01000US|3.31893745E8|2021|
|United States|  01000US|3.33287562E8|2022|
|United States|  01000US|3.34914896E8|2023|
|United States|  01000US| 3.4011099E8|2024|
+-------------+---------+------------+----+



In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW raw_json AS
SELECT *
FROM json.`/Volumes/pr_population_volume/default/raw/data/us_population/population.json`;

SELECT
  exploded_data.*
FROM raw_json
LATERAL VIEW explode(data) AS exploded_data


Nation,Nation ID,Population,Year
United States,01000US,3.16128839E8,2013
United States,01000US,3.18857056E8,2014
United States,01000US,3.21418821E8,2015
United States,01000US,3.23127515E8,2016
United States,01000US,3.25719178E8,2017
United States,01000US,3.27167439E8,2018
United States,01000US,3.28239523E8,2019
United States,01000US,3.31893745E8,2021
United States,01000US,3.33287562E8,2022
United States,01000US,3.34914896E8,2023


In [0]:
from pyspark.sql.functions import col, explode

# Read the JSON file
raw_df = spark.read.json("/Volumes/pr_population_volume/default/raw/data/us_population/population.json")

# Explode the 'data' array and flatten the struct
result_df = raw_df.select(explode(col("data")).alias("exploded_data")).select("exploded_data.*")

result_df.display()


In [0]:

%sql
USE CATALOG pr_population_volume;

SELECT
  exploded_data.Nation as Nation,
  exploded_data.`Nation ID` as Nation_id,
  CAST(exploded_data.Population AS BIGINT) as Population,
  CAST(exploded_data.Year AS INT) as Year   
FROM bronze.bz_us_population
LATERAL VIEW explode(data) AS exploded_data;

Nation,Nation_id,Population,Year
United States,01000US,316128839,2013
United States,01000US,318857056,2014
United States,01000US,321418821,2015
United States,01000US,323127515,2016
United States,01000US,325719178,2017
United States,01000US,327167439,2018
United States,01000US,328239523,2019
United States,01000US,331893745,2021
United States,01000US,333287562,2022
United States,01000US,334914896,2023


In [0]:
%sql
select * from  bronze.bz_us_population;

annotations,columns,data,page,_rescued_data
null,null,null,null,null
null,null,null,null,null
null,null,null,null,null
null,null,null,null,null
null,null,null,null,null
null,null,null,null,null
null,null,null,null,null
null,null,null,null,null
null,null,null,null,null
null,null,null,null,null


In [0]:
%sql


select series_id,cast(year as INT) as Year,period,cast(value as decimal) as value, coalesce(footnote_codes, "NA") as footnote_codes from bronze.bz_bls_pr group by series_id,year,period,value,footnote_codes

series_id,Year,period,value,footnote_codes
PRS30006011,1996,Q02,0,NA
PRS30006011,1996,Q03,0,NA
PRS30006011,1999,Q04,-2,NA
PRS30006011,2002,Q03,-7,NA
PRS30006011,2002,Q04,-5,NA
PRS30006011,2003,Q04,-4,NA
PRS30006011,2007,Q04,-2,NA
PRS30006011,2014,Q01,1,NA
PRS30006011,2016,Q03,0,NA
PRS30006011,2018,Q04,2,NA


In [0]:
%sql
select * from silver.sv_bls_pr

series_id,year,period,value,footnote_codes,processed_tms
PRS32006011,2026,Q02,-1.00,R,2026-10-07T05:24:00.817Z
PRS84006081,2026,Q02,11.50,R,2026-10-07T05:24:00.817Z
PRS30006011,2017,Q02,0.70,-,2026-10-07T05:24:00.817Z
PRS30006013,1988,Q02,143.60,-,2026-10-07T05:24:00.817Z
PRS30006021,2007,Q05,0.10,-,2026-10-07T05:24:00.817Z
PRS30006022,2022,Q04,-2.90,-,2026-10-07T05:24:00.817Z
PRS30006031,2019,Q03,0.00,-,2026-10-07T05:24:00.817Z
PRS30006031,2025,Q04,-0.40,-,2026-10-07T05:24:00.817Z
PRS30006033,2006,Q03,112.41,-,2026-10-07T05:24:00.817Z
PRS30006033,2024,Q05,100.16,-,2026-10-07T05:24:00.817Z


In [0]:
%sql
SELECT
  bls.series_id as Series_ID,
  bls.year  as Year,  
  sum(bls.value) as Value,
  bls.footnote_codes as Footnote_code,  
  coalesce(us_pop.Population,0) as Population 
FROM
  silver.sv_bls_pr as bls
  left join
   silver.sv_us_population_flattened as us_pop
on bls.Year=us_pop.Year
where 
bls.period in ('Q01','Q02','Q03','Q04') 
 and 
bls.series_id='PRS30006032'
group by    
  bls.series_id,
  bls.year,  
  bls.footnote_codes,  
  us_pop.Population
  order by Year

Series_ID,Year,Value,Footnote_code,Population
PRS30006032,1987,9.20,-,0
PRS30006032,1988,8.80,-,0
PRS30006032,1989,-5.50,-,0
PRS30006032,1990,-13.70,-,0
PRS30006032,1991,-8.70,-,0
PRS30006032,1992,-2.00,-,0
PRS30006032,1993,5.70,-,0
PRS30006032,1994,14.60,-,0
PRS30006032,1995,-10.80,-,0
PRS30006032,1996,10.00,-,0


In [0]:
%sql
select * from silver.sv_us_population_flattened

Nation,Nation_id,Population,Year
United States,01000US,316128839,2013
United States,01000US,318857056,2014
United States,01000US,321418821,2015
United States,01000US,323127515,2016
United States,01000US,325719178,2017
United States,01000US,327167439,2018
United States,01000US,328239523,2019
United States,01000US,331893745,2021
United States,01000US,333287562,2022
United States,01000US,334914896,2023


In [0]:
%sql
select
      trim(series_id)   as series_id,
      cast(year as INT) as Year,
      trim(period) as period,
      cast(value as decimal(5,2)) as value,
      coalesce(footnote_codes, "-") as footnote_codes
    from STREAM
      bronze.bz_bls_pr
    group by
      series_id,
      year,
      period,
      value,
      footnote_codes

series_id,Year,period,value,footnote_codes
PRS84006132,2026,Q02,14.80,R
PRS30006032,2002,Q01,-6.60,-
PRS30006033,2011,Q04,95.04,-
PRS30006063,1995,Q02,69.87,-
PRS30006063,1998,Q01,77.85,-
PRS30006063,1998,Q03,79.48,-
PRS30006063,2022,Q04,120.88,-
PRS30006092,2000,Q02,3.10,-
PRS30006102,2017,Q01,3.70,-
PRS30006102,2022,Q05,2.70,-


In [0]:
%sql
drop table silver.sv_bls_pr


In [0]:
%sql
select * from gold.gd_q3_bls_pr

Series_ID,Year,Total_Value,Population
PRS30006032,1987,9.20,0
PRS30006032,1988,8.80,0
PRS30006032,1989,-5.50,0
PRS30006032,1990,-13.70,0
PRS30006032,1991,-8.70,0
PRS30006032,1992,-2.00,0
PRS30006032,1993,5.70,0
PRS30006032,1994,14.60,0
PRS30006032,1995,-10.80,0
PRS30006032,1996,10.00,0
